# Identifikasi Warna Kendaraan Bermotor Roda Dua (HSV + SVM)
**Kelompok 19**
- Muhamad Damar Hati / 15240045
- Ananta Bagas Sasena / 15240078
- Mohammad Ardhian Syarifudin / 15240360

Notebook ini mengunduh dataset **Roboflow (`motorcolour/motor-colour`)** format **YOLOv8**, memotong (*crop*) area motor berdasarkan *bounding box*, mengekstrak fitur **HSV**, dan melatih klasifikasi **Support Vector Machine (SVM)**.

## Sel 1: Download Dataset `motor-colour` dari Roboflow

In [ ]:
!pip install roboflow pyyaml scikit-learn opencv-python matplotlib
from roboflow import Roboflow
rf = Roboflow(api_key="wcJ4wkxJ9USABcf5EJNR")
project = rf.workspace("motorcolour").project("motor-colour")
version = project.version(1)
dataset = version.download("yolov8")

## Sel 2: Crop Bounding Box Motor, Ekstraksi HSV & Training SVM
Setiap gambar di-crop tepat pada koordinat kotak motornya, dikonversi ke **HSV**, lalu dilatih dengan **SVM**.

In [ ]:
import os
import glob
import yaml
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

dataset_dir = dataset.location

with open(os.path.join(dataset_dir, "data.yaml"), "r") as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml["names"]
print("Kelas warna pada dataset:", class_names)

def extract_hsv_from_bbox(img, bbox_norm):
    h, w, _ = img.shape
    xc, yc, bw, bh = bbox_norm
    x1 = max(0, int((xc - bw / 2) * w))
    y1 = max(0, int((yc - bh / 2) * h))
    x2 = min(w, int((xc + bw / 2) * w))
    y2 = min(h, int((yc + bh / 2) * h))
    
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    
    # Ambil 60% bagian tengah kotak agar fokus pada bodi motor
    ch, cw, _ = crop.shape
    body = crop[int(ch*0.2):int(ch*0.8), int(cw*0.2):int(cw*0.8)]
    if body.size == 0:
        body = crop
        
    hsv = cv2.cvtColor(body, cv2.COLOR_BGR2HSV)
    mean_hsv = np.mean(hsv, axis=(0, 1))
    std_hsv = np.std(hsv, axis=(0, 1))
    return np.concatenate([mean_hsv, std_hsv])

X, y = [], []
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_dir, split, "images")
    lbl_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.exists(img_dir):
        continue
    for img_path in glob.glob(os.path.join(img_dir, "*.*")):
        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, base + ".txt")
        if not os.path.exists(lbl_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        with open(lbl_path, "r") as lf:
            for line in lf:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    bbox = [float(v) for v in parts[1:5]]
                    feat = extract_hsv_from_bbox(img, bbox)
                    if feat is not None:
                        X.append(feat)
                        y.append(class_names[cls_id])

X = np.array(X)
y = np.array(y)
print(f"\nTotal objek motor berhasil diekstrak: {len(X)} sampel")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, probability=True, random_state=42))
model_svm.fit(X_train, y_train)

y_pred = model_svm.predict(X_test)
print(f"\nAkurasi SVM: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

## Sel 3: Tes Prediksi dengan Upload Foto Motor Baru

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

uploaded = files.upload()

for fn in uploaded.keys():
    img = cv2.imread(fn)
    feat = extract_hsv_from_bbox(img, [0.5, 0.5, 0.7, 0.7])
    pred = model_svm.predict([feat])[0]
    conf = np.max(model_svm.predict_proba([feat])) * 100

    h_deg = int(feat[0] * 2)
    s_pct = int((feat[1] / 255) * 100)
    v_pct = int((feat[2] / 255) * 100)

    plt.figure(figsize=(5, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"Prediksi SVM: {pred} ({conf:.1f}%)\nHSV: H={h_deg}°, S={s_pct}%, V={v_pct}%", fontsize=12, fontweight="bold")
    plt.show()